# Station Candidate Selection

This notebook prepares the station anchor table used for all urban-context feature extraction. The source is the Transport for NSW station metadata file, not the hourly traffic-count tables.


In [ ]:
import pandas as pd
from pathlib import Path


## Paths

The raw station-reference file should be placed in the project root. The cleaned candidate table is written to the project root so it can be reused by the urban-context extraction notebook.


In [ ]:
input_path = Path("road_traffic_counts_station_reference.csv")
output_path = Path("sydney_permanent_stations_candidate_1.csv")


## Load Required Columns

Only station metadata fields needed for filtering, spatial buffering, and later joins are loaded.


In [ ]:
important_columns = [
    "station_key",
    "station_id",
    "name",
    "road_name",
    "intersection",
    "rms_region",
    "lga",
    "suburb",
    "post_code",
    "device_type",
    "permanent_station",
    "quality_rating",
    "publish",
    "wgs84_latitude",
    "wgs84_longitude",
]

stations_raw = pd.read_csv(input_path, usecols=important_columns)


## Standardise Filter Fields

The source file may store boolean and numeric fields as strings, so the filter columns are normalised before selecting candidate stations.


In [ ]:
stations_raw["rms_region"] = stations_raw["rms_region"].astype("string").str.strip()

for column in [
    "permanent_station",
    "quality_rating",
    "wgs84_latitude",
    "wgs84_longitude",
]:
    stations_raw[column] = pd.to_numeric(stations_raw[column], errors="coerce")

publish_true_values = {"true", "1", "yes", "y"}
stations_raw["publish_clean"] = (
    stations_raw["publish"]
    .astype("string")
    .str.strip()
    .str.lower()
    .isin(publish_true_values)
)


## Select Sydney Permanent Stations

A station is retained when it is in the Sydney RMS region, is a permanent monitoring station, is published, and has a quality rating of at least 4.


In [ ]:
sydney_stations = stations_raw.loc[
    stations_raw["rms_region"].eq("Sydney")
    & stations_raw["permanent_station"].eq(1)
    & stations_raw["publish_clean"]
    & stations_raw["quality_rating"].ge(4)
].copy()

sydney_stations = sydney_stations.dropna(
    subset=["wgs84_latitude", "wgs84_longitude"]
).copy()


## Export Candidate Table

The final table keeps one row per station key and retains only the fields needed by downstream spatial and traffic-data workflows.


In [ ]:
station_columns = [
    "station_key",
    "station_id",
    "name",
    "road_name",
    "intersection",
    "lga",
    "suburb",
    "post_code",
    "device_type",
    "wgs84_latitude",
    "wgs84_longitude",
]

sydney_permanent_stations_candidate_1 = (
    sydney_stations[station_columns]
    .drop_duplicates(subset="station_key")
    .copy()
)

assert sydney_permanent_stations_candidate_1["station_key"].is_unique
assert not sydney_permanent_stations_candidate_1[
    ["wgs84_latitude", "wgs84_longitude"]
].isna().any().any()

sydney_permanent_stations_candidate_1.to_csv(output_path, index=False)

sydney_permanent_stations_candidate_1.shape
